# Thực Hành CNN — 3 Bài Phân Loại Ảnh

| # | Bài | Dataset | Nguồn |
|---|-----|---------|-------|
| 1 | CIFAR-10 (10 lớp) | 60 000 ảnh 32×32 RGB | `keras.datasets` |
| 2 | Dogs vs. Cats (2 lớp) | 23 000 ảnh thực tế | `tensorflow_datasets` |
| 3 | Fashion-MNIST (10 lớp) | 70 000 ảnh 28×28 grayscale | `keras.datasets` |

## Cài đặt chung

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")

---
# Bai 1 — CIFAR-10 (10 lop)

**Nhan:**
| Label | Ten | Label | Ten |
|-------|-----|-------|-----|
| 0 | airplane | 5 | dog |
| 1 | automobile | 6 | frog |
| 2 | bird | 7 | horse |
| 3 | cat | 8 | ship |
| 4 | deer | 9 | truck |

### 1.1 — Load du lieu

In [ ]:
CLASS_NAMES_CIFAR = [
    'airplane','automobile','bird','cat','deer',
    'dog','frog','horse','ship','truck'
]

(X_train_c, y_train_c), (X_test_c, y_test_c) = keras.datasets.cifar10.load_data()

print(f"Train : {X_train_c.shape}  |  Test : {X_test_c.shape}")

In [ ]:
# Hien thi anh mau
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
fig.suptitle('CIFAR-10 — Anh mau (moi cot = 1 lop)', fontsize=13, fontweight='bold')
for cls in range(10):
    idxs = np.where(y_train_c.flatten() == cls)[0][:3]
    for row, idx in enumerate(idxs):
        axes[row, cls].imshow(X_train_c[idx])
        axes[row, cls].axis('off')
        if row == 0:
            axes[row, cls].set_title(CLASS_NAMES_CIFAR[cls], fontsize=8)
plt.tight_layout()
plt.show()

### 1.2 — Tien xu ly

In [ ]:
X_train_c = X_train_c.astype('float32') / 255.0
X_test_c  = X_test_c.astype('float32')  / 255.0

y_train_c_ohe = keras.utils.to_categorical(y_train_c, 10)
y_test_c_ohe  = keras.utils.to_categorical(y_test_c,  10)

print(f"X_train : {X_train_c.shape}  dtype={X_train_c.dtype}")
print(f"y_train : {y_train_c_ohe.shape} (one-hot)")

### 1.3 — Kien truc CNN

```
Input (32x32x3)
 Block 1 : Conv(32) -> BN -> ReLU -> Conv(32) -> BN -> ReLU -> MaxPool -> Dropout(0.25)
 Block 2 : Conv(64) -> BN -> ReLU -> Conv(64) -> BN -> ReLU -> MaxPool -> Dropout(0.25)
 Block 3 : Conv(128)-> BN -> ReLU -> Conv(128)-> BN -> ReLU -> MaxPool -> Dropout(0.25)
 Head    : Flatten -> Dense(256) -> BN -> ReLU -> Dropout(0.5) -> Dense(10) -> Softmax
```

In [ ]:
def build_cifar_cnn():
    model = models.Sequential(name='CIFAR10_CNN')
    model.add(layers.Input(shape=(32, 32, 3)))

    # Data augmentation
    model.add(layers.RandomFlip('horizontal'))
    model.add(layers.RandomRotation(0.1))
    model.add(layers.RandomZoom(0.1))

    for filters in [32, 64, 128]:
        model.add(layers.Conv2D(filters, (3, 3), padding='same'))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation('relu'))
        model.add(layers.Conv2D(filters, (3, 3), padding='same'))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation('relu'))
        model.add(layers.MaxPooling2D((2, 2)))
        model.add(layers.Dropout(0.25))

    model.add(layers.Flatten())
    model.add(layers.Dense(256))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(10, activation='softmax'))

    return model

cifar_model = build_cifar_cnn()
cifar_model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
cifar_model.summary()

### 1.4 — Huan luyen

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_cifar10.keras', monitor='val_accuracy', save_best_only=True, verbose=0)
]

hist_cifar = cifar_model.fit(
    X_train_c, y_train_c_ohe,
    batch_size=64,
    epochs=50,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

### 1.5 — Danh gia

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('CIFAR-10 — Lich su huan luyen', fontsize=13, fontweight='bold')
for ax, (train_key, val_key), title in zip(
    axes,
    [('loss','val_loss'), ('accuracy','val_accuracy')],
    ['Loss', 'Accuracy']
):
    ax.plot(hist_cifar.history[train_key], label='Train', color='royalblue')
    ax.plot(hist_cifar.history[val_key],   label='Val',   color='tomato', linestyle='--')
    ax.set_title(title); ax.set_xlabel('Epochs'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

loss, acc = cifar_model.evaluate(X_test_c, y_test_c_ohe, verbose=0)
print(f"Test Accuracy : {acc*100:.2f}%  |  Test Loss : {loss:.4f}")

In [ ]:
# Confusion matrix
y_pred = np.argmax(cifar_model.predict(X_test_c, verbose=0), axis=1)
y_true = y_test_c.flatten()

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES_CIFAR, yticklabels=CLASS_NAMES_CIFAR)
plt.title('Confusion Matrix — CIFAR-10')
plt.xlabel('Du doan'); plt.ylabel('Thuc te')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES_CIFAR))

---
# Bai 2 — Dogs vs. Cats (Phan loai nhi phan)

Dataset lay truc tiep tu `tensorflow_datasets` (cats_vs_dogs).
~23 000 anh thuc te, tu dong chia 80/10/10 cho train/val/test.

**Output:** Sigmoid + Binary Crossentropy (2 lop: cat=0, dog=1)

### 2.1 — Cai dat tensorflow-datasets

In [ ]:
# Cai tensorflow-datasets neu chua co
# !pip install tensorflow-datasets -q

import tensorflow_datasets as tfds

# Load cats_vs_dogs, chia 80/10/10
(ds_train_raw, ds_val_raw, ds_test_raw), info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    as_supervised=True,
    with_info=True
)

print(f"So mau train : {len(ds_train_raw)}")
print(f"So mau val   : {len(ds_val_raw)}")
print(f"So mau test  : {len(ds_test_raw)}")
print(f"Nhan         : {info.features['label'].names}")   # ['cat', 'dog']

### 2.2 — Pipeline du lieu

In [ ]:
IMG_SIZE   = 128
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.1)
    image = tf.image.random_contrast(image, 0.9, 1.1)
    return image, label

AUTOTUNE = tf.data.AUTOTUNE

ds_train = (ds_train_raw
            .map(preprocess, num_parallel_calls=AUTOTUNE)
            .map(augment,    num_parallel_calls=AUTOTUNE)
            .shuffle(1000)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

ds_val = (ds_val_raw
          .map(preprocess, num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(AUTOTUNE))

ds_test = (ds_test_raw
           .map(preprocess, num_parallel_calls=AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(AUTOTUNE))

In [ ]:
# Hien thi anh mau
CLASS_NAMES_CD = ['cat', 'dog']
sample_batch = next(iter(ds_train))
images, labels = sample_batch

fig, axes = plt.subplots(3, 8, figsize=(14, 6))
fig.suptitle('Dogs vs. Cats — Anh mau sau augmentation', fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i].numpy())
        ax.set_title(CLASS_NAMES_CD[int(labels[i])], fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

### 2.3 — Kien truc CNN

```
Input (128x128x3)
 Block 1 : Conv(32)  -> BN -> ReLU -> MaxPool -> Dropout(0.25)
 Block 2 : Conv(64)  -> BN -> ReLU -> MaxPool -> Dropout(0.25)
 Block 3 : Conv(128) -> BN -> ReLU -> MaxPool -> Dropout(0.25)
 Block 4 : Conv(256) -> BN -> ReLU -> MaxPool -> Dropout(0.25)
 Head    : Flatten -> Dense(512) -> BN -> ReLU -> Dropout(0.5) -> Dense(1) -> Sigmoid
```

In [ ]:
def build_catdog_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    model = models.Sequential(name='CatDog_CNN')
    model.add(layers.Input(shape=input_shape))

    for filters in [32, 64, 128, 256]:
        model.add(layers.Conv2D(filters, (3, 3), padding='same'))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation('relu'))
        model.add(layers.MaxPooling2D((2, 2)))
        model.add(layers.Dropout(0.25))

    model.add(layers.Flatten())
    model.add(layers.Dense(512))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))   # Binary output

    return model

catdog_model = build_catdog_cnn()
catdog_model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
catdog_model.summary()

### 2.4 — Huan luyen

In [ ]:
callbacks_cd = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_catdog.keras', monitor='val_accuracy', save_best_only=True, verbose=0)
]

hist_cd = catdog_model.fit(
    ds_train,
    epochs=30,
    validation_data=ds_val,
    callbacks=callbacks_cd,
    verbose=1
)

### 2.5 — Danh gia

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Dogs vs. Cats — Lich su huan luyen', fontsize=13, fontweight='bold')
for ax, (train_key, val_key), title in zip(
    axes,
    [('loss','val_loss'), ('accuracy','val_accuracy')],
    ['Loss', 'Accuracy']
):
    ax.plot(hist_cd.history[train_key], label='Train', color='royalblue')
    ax.plot(hist_cd.history[val_key],   label='Val',   color='tomato', linestyle='--')
    ax.set_title(title); ax.set_xlabel('Epochs'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

loss, acc = catdog_model.evaluate(ds_test, verbose=0)
print(f"Test Accuracy : {acc*100:.2f}%  |  Test Loss : {loss:.4f}")

In [ ]:
# Confusion matrix tren tap test
y_pred_list, y_true_list = [], []
for images, labels in ds_test:
    preds = catdog_model.predict(images, verbose=0).flatten()
    y_pred_list.extend((preds > 0.5).astype(int))
    y_true_list.extend(labels.numpy())

y_pred_cd = np.array(y_pred_list)
y_true_cd = np.array(y_true_list)

cm_cd = confusion_matrix(y_true_cd, y_pred_cd)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_cd, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASS_NAMES_CD, yticklabels=CLASS_NAMES_CD)
plt.title('Confusion Matrix — Dogs vs. Cats')
plt.xlabel('Du doan'); plt.ylabel('Thuc te')
plt.tight_layout(); plt.show()

print(classification_report(y_true_cd, y_pred_cd, target_names=CLASS_NAMES_CD))

In [ ]:
# Hien thi du doan tren anh thuc te
sample_images, sample_labels = next(iter(ds_test))
preds = catdog_model.predict(sample_images, verbose=0).flatten()

fig, axes = plt.subplots(3, 8, figsize=(14, 6))
fig.suptitle('Dogs vs. Cats — Du doan (Xanh=Dung, Do=Sai)', fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flat):
    if i < len(sample_images):
        ax.imshow(sample_images[i].numpy())
        pred_label = 'dog' if preds[i] > 0.5 else 'cat'
        true_label = CLASS_NAMES_CD[int(sample_labels[i])]
        conf = preds[i] if pred_label == 'dog' else 1 - preds[i]
        color = 'green' if pred_label == true_label else 'red'
        ax.set_title(f'P:{pred_label}\n({conf:.0%}) T:{true_label}', color=color, fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

---
# Bai 3 — Fashion-MNIST (10 lop)

Dataset lay truc tiep tu `keras.datasets.fashion_mnist` (~26MB).
Anh 28×28 grayscale (1 kenh mau).

**Nhan:**
| Label | Ten | Label | Ten |
|-------|-----|-------|-----|
| 0 | T-shirt/top | 5 | Sandal |
| 1 | Trouser | 6 | Shirt |
| 2 | Pullover | 7 | Sneaker |
| 3 | Dress | 8 | Bag |
| 4 | Coat | 9 | Ankle boot |

### 3.1 — Load du lieu

In [ ]:
CLASS_NAMES_FM = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

(X_train_fm, y_train_fm), (X_test_fm, y_test_fm) = keras.datasets.fashion_mnist.load_data()

print(f"Train : {X_train_fm.shape}  |  Test : {X_test_fm.shape}")
print(f"Pixel : [{X_train_fm.min()}, {X_train_fm.max()}]")

In [ ]:
# Hien thi anh mau
fig, axes = plt.subplots(2, 10, figsize=(15, 4))
fig.suptitle('Fashion-MNIST — Anh mau (moi cot = 1 lop)', fontsize=13, fontweight='bold')
for cls in range(10):
    idxs = np.where(y_train_fm == cls)[0][:2]
    for row, idx in enumerate(idxs):
        axes[row, cls].imshow(X_train_fm[idx], cmap='gray_r')
        axes[row, cls].axis('off')
        if row == 0:
            axes[row, cls].set_title(CLASS_NAMES_FM[cls], fontsize=7, rotation=20)
plt.tight_layout(); plt.show()

### 3.2 — Tien xu ly

In [ ]:
# Them chieu channel: (N,28,28) -> (N,28,28,1)
X_train_fm = X_train_fm[..., np.newaxis].astype('float32') / 255.0
X_test_fm  = X_test_fm[..., np.newaxis].astype('float32')  / 255.0

y_train_fm_ohe = keras.utils.to_categorical(y_train_fm, 10)
y_test_fm_ohe  = keras.utils.to_categorical(y_test_fm,  10)

print(f"X_train : {X_train_fm.shape}  dtype={X_train_fm.dtype}")
print(f"y_train : {y_train_fm_ohe.shape} (one-hot)")

### 3.3 — Kien truc CNN

```
Input (28x28x1)  <- Grayscale: 1 kenh mau
 Block 1 : Conv(32) -> BN -> ReLU -> Conv(32) -> BN -> ReLU -> MaxPool -> Dropout(0.25)
 Block 2 : Conv(64) -> BN -> ReLU -> Conv(64) -> BN -> ReLU -> MaxPool -> Dropout(0.25)
 Block 3 : Conv(128)-> BN -> ReLU                            -> MaxPool -> Dropout(0.25)
 Head    : Flatten -> Dense(256) -> BN -> ReLU -> Dropout(0.5) -> Dense(10) -> Softmax
```

In [ ]:
def build_fashion_cnn(input_shape=(28, 28, 1), num_classes=10):
    model = models.Sequential(name='FashionMNIST_CNN')
    model.add(layers.Input(shape=input_shape))

    # Augmentation nhe cho anh grayscale
    model.add(layers.RandomFlip('horizontal'))
    model.add(layers.RandomRotation(0.05))
    model.add(layers.RandomZoom(0.1))

    # Block 1 & 2: 2 lop conv
    for filters in [32, 64]:
        model.add(layers.Conv2D(filters, (3, 3), padding='same'))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation('relu'))
        model.add(layers.Conv2D(filters, (3, 3), padding='same'))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation('relu'))
        model.add(layers.MaxPooling2D((2, 2)))
        model.add(layers.Dropout(0.25))

    # Block 3: 1 lop conv (anh nho, tranh mat thong tin)
    model.add(layers.Conv2D(128, (3, 3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    model.add(layers.Flatten())
    model.add(layers.Dense(256))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

fashion_model = build_fashion_cnn()
fashion_model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
fashion_model.summary()

### 3.4 — Huan luyen

In [ ]:
callbacks_fm = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_fashion.keras', monitor='val_accuracy', save_best_only=True, verbose=0)
]

hist_fashion = fashion_model.fit(
    X_train_fm, y_train_fm_ohe,
    batch_size=64,
    epochs=50,
    validation_split=0.1,
    callbacks=callbacks_fm,
    verbose=1
)

### 3.5 — Danh gia

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Fashion-MNIST — Lich su huan luyen', fontsize=13, fontweight='bold')
for ax, (train_key, val_key), title in zip(
    axes,
    [('loss','val_loss'), ('accuracy','val_accuracy')],
    ['Loss', 'Accuracy']
):
    ax.plot(hist_fashion.history[train_key], label='Train', color='royalblue')
    ax.plot(hist_fashion.history[val_key],   label='Val',   color='tomato', linestyle='--')
    ax.set_title(title); ax.set_xlabel('Epochs'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

loss, acc = fashion_model.evaluate(X_test_fm, y_test_fm_ohe, verbose=0)
print(f"Test Accuracy : {acc*100:.2f}%  |  Test Loss : {loss:.4f}")

In [ ]:
# Confusion matrix
y_pred_fm = np.argmax(fashion_model.predict(X_test_fm, verbose=0), axis=1)
y_true_fm = y_test_fm.flatten()

cm_fm = confusion_matrix(y_true_fm, y_pred_fm)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_fm, annot=True, fmt='d', cmap='Purples',
            xticklabels=CLASS_NAMES_FM, yticklabels=CLASS_NAMES_FM)
plt.title('Confusion Matrix — Fashion-MNIST')
plt.xlabel('Du doan'); plt.ylabel('Thuc te')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

print(classification_report(y_true_fm, y_pred_fm, target_names=CLASS_NAMES_FM))

In [ ]:
# Hien thi du doan
indices = np.random.choice(len(X_test_fm), 20, replace=False)
y_pred_prob_fm = fashion_model.predict(X_test_fm, verbose=0)

fig, axes = plt.subplots(4, 5, figsize=(13, 11))
fig.suptitle('Fashion-MNIST — Du doan (Xanh=Dung, Do=Sai)', fontsize=13, fontweight='bold')
for ax, idx in zip(axes.flat, indices):
    ax.imshow(X_test_fm[idx].squeeze(), cmap='gray_r')
    pred = np.argmax(y_pred_prob_fm[idx])
    true = y_true_fm[idx]
    conf = y_pred_prob_fm[idx][pred]
    color = 'green' if pred == true else 'red'
    ax.set_title(f'P: {CLASS_NAMES_FM[pred]}\n({conf:.0%}) T: {CLASS_NAMES_FM[true]}',
                 color=color, fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

---
# Tong ket 3 Bai

| | Bai 1: CIFAR-10 | Bai 2: Dogs vs. Cats | Bai 3: Fashion-MNIST |
|-|-----------------|----------------------|----------------------|
| **Nguon** | `keras.datasets` | `tensorflow_datasets` | `keras.datasets` |
| **Kich thuoc** | 32x32x3 | 128x128x3 | 28x28x1 |
| **So lop** | 10 | 2 | 10 |
| **Output** | Softmax | Sigmoid | Softmax |
| **Loss** | Categorical CE | Binary CE | Categorical CE |
| **Accuracy ky vong** | ~87% | ~90%+ | ~92%+ |

**Diem khac biet chinh:**
- Bai 2 dung `Sigmoid + binary_crossentropy` vi chi co 2 lop.
- Bai 3 anh grayscale (1 kenh) nen them `[..., np.newaxis]` khi tien xu ly.
- Bai 2 dung `tf.data.Dataset` pipeline thay vi `ImageDataGenerator` (hieu nang cao hon).
